# Анализ RTT

In [ ]:
! pip install plotly numpy pandas
! pip install --upgrade nbformat
! pip install --upgrade kaleido

In [ ]:
import plotly.express as px
import numpy as np
import pandas as pd
import subprocess
import random
import time

In [ ]:
intervals = [
    10000,
    20000,
    30000,
    40000,
    50000,
    60000,
    70000,
    80000,
    90000,
    100000,
    110000,
    120000,
    130000,
]
ATTEMPTS = 100
REPEATS = 10

### Проведение эксперимента 2

Автоматический прогон всех экспериментов с сохранением данных в папку [data3/](data3/)

In [ ]:
run_parameters = []
for interval in intervals:
    for i in range(REPEATS):
        run_parameters.append((interval, ATTEMPTS, i))

In [ ]:
random.shuffle(run_parameters)

In [ ]:
run_parameters = [(10000, ATTEMPTS, 0)]

In [ ]:
for interval, attempts, num in run_parameters:
    print(interval, "#", num, "start")

    with open(f"./data3/{interval}us#{num}.csv", "w") as f:
        try:
            subprocess.run(
                ["./client/ClientProject", "192.168.0.101", "8080", str(interval), str(attempts)], 
                timeout=20,
                stdout=f, 
                check=True)
        except subprocess.TimeoutExpired as e:
            print(f"Command timed out after {e.timeout} seconds.")

    print(interval, "#", num, "finish")

    time.sleep(2)


## График RTT от попытки отправки пакета

In [ ]:
for interval in intervals:
    for repeat in range(REPEATS):
        print(f"./data3/{interval}us#{repeat}.csv")
        df = pd.read_csv(f"./data3/{interval}us#{repeat}.csv")
        fig = px.line(
            df,
            x=df.index,
            y="rtt",
            color="interval",
            title=f"RTT to attempt with interval")
        fig.show()

In [ ]:
dfs = []
for interval in intervals:
    df = pd.read_csv(f"./data2/{interval}us.csv")
    dfs.append(df)

df = pd.concat(dfs)
fig = px.line(
    df,
    x=df.index,
    y="rtt",
    color="interval",
    title=f"RTT to attempt with interval")
fig.show()

## График среднего RTT на последних двадцати попытках

In [ ]:
ATTEMPTS = 1000
SMOOTH_ATTEMPTS = 20

dfs = []

for interval in intervals:
    df = pd.read_csv(f"./data2/{interval}us.csv")
    rtt_cumsum = np.cumsum(df['rtt']) 
    rtt_cumsum_attempts_before = np.append(np.zeros(SMOOTH_ATTEMPTS), np.cumsum(df['rtt'])[:(ATTEMPTS - SMOOTH_ATTEMPTS)])
    df['rtt'] = (rtt_cumsum - rtt_cumsum_attempts_before) / SMOOTH_ATTEMPTS
    dfs.append(df)

df = pd.concat(dfs)
fig = px.line(
    df,
    x=df.index,
    y="rtt",
    color="interval",
    title=f"RTT to attempt with interval")
fig.show()

## Среднее значение RTT на каждом эксперименте

In [ ]:
dfs = {
    "interval": [],
    "rtt_average": [],
    "rtt_error": [],
}

for interval in intervals:
    df = pd.read_csv(f"./data2/{interval}us.csv")
    dfs['rtt_average'].append(np.average(df['rtt']))
    dfs['rtt_error'].append(np.std(df['rtt']))
    dfs['interval'].append(interval)

df = pd.DataFrame(dfs)
fig = px.line(
    df,
    x="interval",
    y="rtt_average",
    error_y="rtt_error",
    title=f"RTT average to interval")
fig.show()

### Эксперимент 2

Среднее значение RTT на каждом подксперименте для каждого интервала

In [ ]:
dfs = {
    "interval": [],
    "rtt_average": [],
    "rtt_error": [],
}

for interval in intervals:
    averages = []
    for repeat in range(REPEATS):
        df = pd.read_csv(f"./data3/{interval}us#{repeat}.csv")
        averages.append(np.average(df['rtt']))
    dfs['rtt_average'].append(np.average(averages))
    dfs['rtt_error'].append(np.std(averages))
    dfs['interval'].append(interval)

df = pd.DataFrame(dfs)
fig = px.line(
    df,
    x="interval",
    y="rtt_average",
    error_y="rtt_error",
    title=f"RTT average to interval")
fig.show()

Среднее значение RTT на каждом подксперименте для каждого интервала с отброшенными начальными нестабильными пакетами (10 штук)

In [ ]:
dfs = {
    "interval": [],
    "rtt_average": [],
    "rtt_error": [],
}

for interval in intervals:
    averages = []
    for repeat in range(REPEATS):
        df = pd.read_csv(f"./data3/{interval}us#{repeat}.csv")
        averages.append(np.average(df['rtt'][10:]))
    dfs['rtt_average'].append(np.average(averages))
    dfs['rtt_error'].append(np.std(averages))
    dfs['interval'].append(interval)

df = pd.DataFrame(dfs)
fig = px.line(
    df,
    x="interval",
    y="rtt_average",
    error_y="rtt_error",
    title=f"RTT average to interval")
fig.show()

Подсчёт количества таких подэкспериментов, в котором есть начальные нестабильные пакеты

In [ ]:
counters = {}
for interval in intervals:
    counters[interval] = 0

for interval in intervals:
    averages = []
    for repeat in range(REPEATS):
        df = pd.read_csv(f"./data3/{interval}us#{repeat}.csv")
        if df['rtt'][0] > 30000 and np.average(df['rtt']) < 10000:
            counters[interval] += 1
counters